# ETL Transform: Stocks

This notebook runs the **stocks ETL pipeline**: ingest from Postgres (with warmup window) → transform (returns, volatility, technical indicators) → save to `historical_processed` → publish to S3 per book/day.

**S3 path**: `stocks/crypto/book={book}/year=.../month=.../day=.../format=csv/YYYYMMDD-{book}.csv`

Set `AWS_STOCKS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads.

In [ ]:
import sys
from pathlib import Path

# Resolve project root: works whether kernel cwd is DataProcessing/ or repo root
_cwd = Path(".").resolve()
if _cwd.name == "DataProcessing":
    project_root = _cwd.parent
else:
    project_root = _cwd
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run this notebook from repo root or DataProcessing/.")

import pandas as pd

In [ ]:
# Run stocks ETL: transform (with warmup for indicators), save to Postgres, publish to S3.

from pipelines.etl_transform import run_stocks_etl

transformed_df = run_stocks_etl(
    since="2025-01-01",
    until="2025-12-31",
    books=None,  # all books; or e.g. ["btc-usd", "eth-usd"]
    warmup_days=252,
    stocks_bucket=None,  # uses AWS_STOCKS_BUCKET or AWS_DEFAULT_BUCKET from .env
    save_to_postgres=True,
    upload_s3=True,
)

print(f"Transformed {len(transformed_df)} stock records")

In [ ]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

## Optional: Specific books and date range

In [ ]:
# transformed_df = run_stocks_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     books=["btc-usd", "eth-usd"],
#     warmup_days=252,
#     save_to_postgres=True,
#     upload_s3=True,
# )